In [ ]:
import os
import pandas as pd
from datetime import datetime
import seaborn as sns
import math
import numpy as np
from collections import defaultdict
from tqdm import tqdm

In [68]:
path='./archive/dressipi_recsys2022/'
train_sessions = pd.read_csv(path+"train_sessions.csv")
train_purchase = pd.read_csv(path+"train_purchases.csv")
item_features = pd.read_csv(path+"item_features.csv")
test_leaderboard_sessions = pd.read_csv(path+"test_leaderboard_sessions.csv")
test_final_sessions=pd.read_csv(path+"test_final_sessions.csv")

In [69]:
train_sessions=train_sessions.append(train_purchase)

C:\Users\Smike\AppData\Local\Temp\ipykernel_53968\3479522542.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  train_sessions=train_sessions.append(train_purchase)


In [70]:
user_item_dict=train_sessions.groupby("session_id")["item_id"].agg(list).to_dict()
user_item_dict
sim_item={}
item_cnt=defaultdict(int)
for user,items in tqdm(user_item_dict.items()):
    for i in items:
        item_cnt[i]+=1
        sim_item.setdefault(i,{})
        for relate_item in items:
            if i == relate_item:
                continue
            sim_item[i].setdefault(relate_item,0)
            #防止session长度过长导致的问题,len(items)是当前session的长度
            #对短session类似于一种奖励机制
            sim_item[i][relate_item]+=1/math.log(1+len(items))
            

    

100%|██████████| 1000000/1000000 [01:01<00:00, 16201.32it/s]


In [71]:
sim_item_corr=sim_item.copy()

for i,relate_item in tqdm(sim_item.items()):
    for j,cij in relate_item.items():
        #相似矩阵归一化，解决热门物品出现次数过多的问题
        
        sim_item_corr[i][j]= cij/math.sqrt(item_cnt[i]*item_cnt[j])
        

100%|██████████| 23618/23618 [00:15<00:00, 1546.82it/s] 


In [72]:
sim_item_corr[4]

{19718: 0.05494469074128063,
 1422: 0.007944756834535303,
 1020: 0.004972191539123525,
 6893: 0.02121390246771585,
 25415: 0.0070639065666054214,
 16727: 0.001032859984868337,
 1148: 0.005470021317391243,
 18734: 0.007025150018936949,
 8535: 0.006941043157037994,
 18509: 0.0021280487770120294,
 7414: 0.013328779187789075,
 4118: 0.01329552597066934,
 22736: 0.00744186378936747,
 3276: 0.0020050014716258982,
 8914: 0.012718141622704291,
 1735: 0.021716383469405846,
 7365: 0.008316718755725849,
 26653: 0.005347415198790895,
 23612: 0.007269186124316953,
 18969: 0.0004691985312772672,
 21201: 0.002373305612797082,
 17638: 0.003050039061721963,
 5343: 0.004076871759472437,
 14124: 0.0015157858664617925,
 24736: 0.0043653440067362645,
 15458: 0.004570620389980391,
 26132: 0.004435757975831009,
 17086: 0.0008130159402384024,
 19307: 0.007043762163421133,
 26853: 0.01023819235110267,
 26868: 0.002383021821145441,
 7306: 0.0009641561567922064,
 20798: 0.003285984949793379,
 18086: 0.0012986086

#根据相似矩阵推荐topk


In [73]:
#按出现次数统计流行商品
order=train_sessions['item_id'].value_counts()


In [74]:
popular_items=list(order.index)

In [75]:
test_session_dict=test_final_sessions.groupby("session_id")["item_id"].agg(list).to_dict()

In [78]:
session_item_list=test_session_dict
session_item_list

{61: [27088],
 96: [11693, 18298, 4738, 495, 6871],
 185: [17618, 21330, 21330, 17618, 4983],
 224: [24665, 11917],
 285: [15073],
 400: [8060, 23764, 1368, 23764, 8060, 24948],
 580: [16598, 12555, 15249, 26853, 1933, 1933, 15249],
 660: [16157],
 663: [16069],
 792: [27096],
 796: [1818, 8981, 20689, 17362],
 804: [8861, 6322],
 924: [13108, 9046],
 1054: [23322, 11428, 6871, 20127, 793, 20414, 19310, 22279],
 1245: [13642, 13642, 13642],
 1302: [26720],
 1396: [14004, 5226, 27882, 14004, 3717, 14271, 19532],
 1489: [3741, 26644, 13734, 22703, 26402, 25320, 21939],
 1541: [2709, 18451, 12678, 4400, 17059, 13642],
 1691: [7818, 9040, 12678],
 1692: [6129,
  27173,
  1398,
  1398,
  13287,
  15323,
  7399,
  16466,
  25179,
  16718,
  1222,
  7594,
  7669,
  3213,
  13535,
  23002,
  14697,
  16413,
  7248,
  21928,
  6774,
  6774,
  25300,
  1589],
 1732: [8664, 5425, 1714, 6100, 25623, 19270, 3825, 8036],
 1844: [26278, 4492],
 2656: [11044, 22197, 23831, 5205, 21575],
 2728: [24399]

In [80]:
def recommand(session_item_list):
    rank={}
    for i in session_item_list:
        if i not in sim_item_corr:
            continue
        for j,wij in sorted(sim_item_corr[i].items(),key=lambda d:d[1],reverse=True):
            #避免推荐重复物品以及看过的物品
            if j not in session_item_list:
                rank.setdefault(j,0)
                #分数累加
                rank[j]+=wij
    if len(rank)==0:
        item_list=popular_items[:100]
        score_list=0
    else:
        rank=sorted(rank.items(),key=lambda d:d[1],reverse=True)
        rank=np.array(rank)
        item_list=rank[:,0].astype("int32")
        score_list=rank[:,1]
        
        if len(item_list)<100:
            index=0
            while len(item_list)<100:
                item_list.append(popular_items[index])
                item_list=list(set(item_list))
                index+=1
    return list(item_list)

In [81]:
temp=recommand(session_item_list)

In [82]:
session_id_list=[]
item_id_list=[]
rank_list=[]
for session_id,session_item in tqdm(test_session_dict.items()):
    temp_item_list=recommand(session_item_list)
    session_id_list +=[session_id for _ in range(100)]
    item_id_list+=temp_item_list
    rank_list+=[x for x in range(1,101)]
    

 17%|█▋        | 8670/50000 [30:34<2:25:44,  4.73it/s]


MemoryError: 

In [ ]:
res_df=pd.DataFrame()
res_df["session_id"]=session_id_list
res_df["item_id"]=item_id_list
res_df["rank"]=rank_list
res_df.to_csv("baseline.csv",index=False)

ValueError: Length of values (217650000) does not match length of index (5000000)